# AG2 Multi-Agent RAG with txtai

This notebook demonstrates how to use [AG2](https://ag2.ai/) (formerly AutoGen) multi-agent conversations with [txtai](https://neuml.github.io/txtai/) for Retrieval-Augmented Generation (RAG).

**txtai** is an all-in-one AI framework with an embeddings database combining vector indexes, graph networks, and relational databases — enabling semantic search and serving as a powerful knowledge source for LLM applications.

**AG2** is a multi-agent conversation framework with 500K+ monthly PyPI downloads, 4,300+ GitHub stars, and 400+ contributors.

This example shows AG2 as an **alternative multi-agent orchestration layer** for txtai-powered RAG — complementing txtai's built-in agent system with AG2's multi-agent conversation patterns.

Two AG2 agents collaborate:
- **Research Agent**: Retrieves relevant documents from txtai's embeddings database
- **Analyst Agent**: Synthesizes retrieved information into comprehensive answers

# Install dependencies

Install `txtai` and `ag2` with OpenAI support.

In [ ]:
%%capture
!pip install -U txtai "ag2[openai]>=0.11.4,<1.0"

In [ ]:
import os

from autogen import AssistantAgent, GroupChat, GroupChatManager, LLMConfig, UserProxyAgent

import txtai

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "your-api-key"  # Replace with your key

# Create txtai Embeddings Database

Build an in-memory embeddings database with content storage enabled. txtai handles embedding generation internally — no external vector DB or embedding API needed.

In [ ]:
# Create embeddings instance with content storage
# content=True enables returning full document text in search results
embeddings = txtai.Embeddings(content=True)

# Sample documents about AI topics
documents = [
    (
        0,
        "Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with language model generation. "
        "It first retrieves relevant documents from a knowledge base, then uses them as context for generating accurate, grounded "
        "responses. RAG reduces hallucination and enables models to access up-to-date information beyond their training data.",
        None,
    ),
    (
        1,
        "Vector databases store data as high-dimensional vectors (embeddings) and enable fast similarity search. txtai provides an "
        "all-in-one embeddings database that combines vector indexes with graph networks and relational databases, supporting both "
        "semantic and SQL-based queries.",
        None,
    ),
    (
        2,
        "Multi-agent systems use multiple AI agents that collaborate to solve complex tasks. Each agent can have specialized roles, "
        "tools, and knowledge. AG2 (formerly AutoGen) is a popular framework for building multi-agent conversations where agents can "
        "use tools, write code, and coordinate through structured dialogue patterns.",
        None,
    ),
    (
        3,
        "Embedding models convert text into dense numerical vectors that capture semantic meaning. Similar texts produce similar "
        "vectors, enabling semantic search. txtai uses sentence-transformers by default and supports custom embedding models "
        "via Hugging Face.",
        None,
    ),
    (
        4,
        "Chunking is the process of splitting documents into smaller pieces for embedding and retrieval. Common strategies include "
        "fixed-size chunking, recursive character splitting, and semantic chunking. Optimal chunk size depends on the use case: "
        "256-512 tokens for precise retrieval, 1000+ tokens for broader context.",
        None,
    ),
]

# Index documents into txtai
embeddings.index(documents)
print(f"Indexed {len(documents)} documents into txtai embeddings database")

# Test Semantic Search

Verify that txtai retrieval works before connecting it to AG2 agents.

In [ ]:
# Test search — txtai handles embedding + search internally
results = embeddings.search("What is RAG?", limit=3)

for result in results:
    # With content=True, results are dicts with id, score, text
    print(f"Score: {result['score']:.4f}")
    print(f"  {result['text'][:100]}...")
    print()

# Define txtai Search Function for AG2

Create a search function that AG2 agents will use as a registered tool to retrieve relevant documents from the txtai embeddings database.

In [ ]:
def search_txtai(query: str, top_k: int = 3) -> str:
    """
    Search txtai embeddings database for relevant documents.

    Args:
        query: Search query string.
        top_k: Number of results to return.

    Returns:
        Formatted string with retrieved documents and scores.
    """
    results = embeddings.search(query, limit=top_k)

    formatted = []
    for i, result in enumerate(results, 1):
        score = result["score"]
        text = result["text"]
        formatted.append(f"[{i}] (score: {score:.4f})\n{text}")

    return "\n\n---\n\n".join(formatted) if formatted else "No relevant documents found."


# Quick test
print(search_txtai("What is RAG?"))

# Set Up AG2 Multi-Agent System

Create AG2 agents that collaborate to answer questions using txtai retrieval:

1. **Research Agent** — Uses the `search_documents` tool to find relevant documents
2. **Analyst Agent** — Synthesizes retrieved information into a final answer
3. **User Proxy** — Orchestrates the conversation and executes tool calls

In [ ]:
# AG2 LLM Configuration
llm_config = LLMConfig({"model": "gpt-4o-mini", "api_key": os.environ["OPENAI_API_KEY"], "api_type": "openai"})

# Create agents
researcher = AssistantAgent(
    name="researcher",
    system_message=(
        "You are a research agent. When asked a question, use the search_documents "
        "tool to retrieve relevant documents from the txtai knowledge base. Present "
        "your findings clearly with relevance scores. If results are insufficient, "
        "try rephrasing your search query."
    ),
    llm_config=llm_config,
)

analyst = AssistantAgent(
    name="analyst",
    system_message=(
        "You are an analyst. Based on the researcher's findings, synthesize the "
        "information into a comprehensive, well-structured answer. Always reference "
        "the retrieved content. End with TERMINATE when done."
    ),
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config=False,
    is_termination_msg=lambda x: x.get("content", "") and "TERMINATE" in x.get("content", ""),
)


# Register txtai search as AG2 tool
@user_proxy.register_for_execution()
@researcher.register_for_llm(
    description=(
        "Search the txtai embeddings database for relevant documents. "
        "Returns document chunks with similarity scores. "
        "Use specific search queries for best results."
    )
)
def search_documents(query: str, top_k: int = 3) -> str:
    """Search txtai for relevant documents."""
    return search_txtai(query, top_k)


# Set up GroupChat
group_chat = GroupChat(agents=[user_proxy, researcher, analyst], messages=[], max_round=12)

manager = GroupChatManager(groupchat=group_chat, llm_config=llm_config)

# Run Multi-Agent RAG Conversation

The Research Agent will search txtai for relevant documents, and the Analyst Agent will synthesize the findings.

In [ ]:
user_proxy.run(
    manager,
    message="What is RAG and how does it work with embeddings databases? Also explain how multi-agent systems can improve RAG quality.",
).process()

# Summary

This notebook demonstrated:

1. **txtai** as an all-in-one embeddings database for semantic search
2. **AG2** multi-agent conversations with tool-calling capabilities
3. **Combining both**: AG2 agents using txtai search as a registered tool for grounded, retrieval-backed RAG responses

## txtai vs AG2 Agents

txtai includes its own agent system (built on smolagents). This notebook shows AG2 as a **complementary alternative** — useful when you need:
- Multi-agent conversation patterns (GroupChat, speaker selection)
- Integration with AG2's broader ecosystem (code execution, nested chats)
- Compatibility with existing AG2-based workflows

## Key Components

| Component | Role |
|-----------|------|
| **txtai Embeddings** | In-memory vector database with content storage |
| **AG2 UserProxy** | Orchestrates conversation, executes tools |
| **AG2 Research Agent** | Calls `search_documents` tool |
| **AG2 Analyst Agent** | Synthesizes findings into final answer |

## Next Steps

- Scale up with larger document collections and persistent storage
- Add more specialized agents (fact-checker, summarizer, etc.)
- Use txtai SQL queries for structured data retrieval
- Combine txtai pipelines (summarization, translation) as additional AG2 tools
- Explore txtai's graph network features for knowledge graph RAG

## Resources

- [AG2 Documentation](https://docs.ag2.ai/)
- [txtai Documentation](https://neuml.github.io/txtai/)
- [AG2 GitHub](https://github.com/ag2ai/ag2)
- [txtai GitHub](https://github.com/neuml/txtai)